# Ngay 27 Transformer · Colab
Bật GPU nếu có. Notebook tự chọn CUDA, MPS hoặc CPU. Chấp nhận luật cuộc thi trước khi tải. Thực thi và nộp submission thủ công. Cấu hình ít epoch để thử quy trình.

Thêm KAGGLE_JSON vào Colab Secrets và cấp quyền cho notebook. Cài gói kaggle nếu runtime chưa có.

In [ ]:
from pathlib import Path
import os, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
def train_epoch(model, loader, optimizer):
    model.train()
    for x,y in loader:
        optimizer.zero_grad()
        loss=nn.functional.cross_entropy(model(x.to(device)),y.to(device))
        loss.backward()
        optimizer.step()
@torch.inference_mode()
def predict(model, loader):
    model.eval()
    return torch.cat([model(batch[0].to(device)).argmax(1).cpu() for batch in loader])
print('Thiết bị:', device)

In [ ]:
# Lưu JSON của Kaggle trong Colab Secrets dưới tên KAGGLE_JSON.
# Hàm chỉ được gọi khi INPUT_DIR không có dữ liệu và bạn cần tải từ Kaggle.
def download_kaggle(slug, competition=True):
    from google.colab import userdata
    from pathlib import Path
    import os, shutil, subprocess, zipfile
    credentials=Path.home()/'.kaggle/kaggle.json'
    os.environ['KAGGLE_CONFIG_DIR']=str(credentials.parent)
    credentials.parent.mkdir(mode=0o700, exist_ok=True)
    installed=not credentials.exists()
    if installed:
        descriptor=os.open(credentials,os.O_WRONLY|os.O_CREAT|os.O_EXCL,0o600)
        with os.fdopen(descriptor,'w') as handle:
            handle.write(userdata.get('KAGGLE_JSON'))
    try:
        credentials.chmod(0o600)
        cli=shutil.which('kaggle')
        if cli is None:
            raise RuntimeError('Cài Kaggle CLI trong runtime trước khi tải.')
        target=(Path('/content')/slug.rsplit('/',1)[-1]).resolve()
        target.mkdir(exist_ok=True)
        kind,opt=('competitions','-c') if competition else ('datasets','-d')
        subprocess.run([cli,kind,'download',opt,slug,'-p',str(target)],check=True)
        for file in target.glob('*.zip'):
            with zipfile.ZipFile(file) as bundle:
                if any(not (target/member.filename).resolve().is_relative_to(target) for member in bundle.infolist()):
                    raise ValueError('Tệp nén chứa đường dẫn không hợp lệ.')
                bundle.extractall(target)
        return target
    finally:
        if installed:
            credentials.unlink(missing_ok=True)

In [ ]:
ROOT=Path('/content')
INPUT_DIR=download_kaggle('lakshmi25npathi/imdb-dataset-of-50k-movie-reviews',competition=False)

In [ ]:
import re
from collections import Counter

# Kiểm chứng attention đơn đầu với thư viện trước khi dùng Transformer.
seed_all(17)
q,k,v=(torch.randn(2,6,8) for _ in range(3))
out=(q @ k.transpose(-1,-2) / q.shape[-1]**.5).softmax(dim=-1) @ v
torch.testing.assert_close(out, nn.functional.scaled_dot_product_attention(q,k,v))

imdb=pd.read_csv(INPUT_DIR/'IMDB Dataset.csv')
rows=imdb.sample(n=min(2500,len(imdb)),random_state=17).reset_index(drop=True)
train, val=rows.iloc[:-500],rows.iloc[-500:]
def tokens(text):
    return re.findall(r'[a-z]+',text.lower())[:96]
counts=Counter(word for review in train.review for word in tokens(review))
vocab={word:i+2 for i,(word,_) in enumerate(counts.most_common(5000))}
def encode(frame):
    words=[[vocab.get(w,1) for w in tokens(text)] for text in frame.review]
    x=torch.tensor([(row or [1])+[0]*(96-max(1,len(row))) for row in words])
    y=torch.tensor(frame.sentiment.eq('positive').to_numpy(),dtype=torch.long)
    return TensorDataset(x,y)
class ReviewClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding=nn.Embedding(len(vocab)+2,32,padding_idx=0)
        self.position=nn.Embedding(96,32)
        self.encoder=nn.TransformerEncoderLayer(32,4,64,batch_first=True)
        self.head=nn.Linear(32,2)
    def forward(self,x):
        pad=x.eq(0)
        h=self.embedding(x)+self.position(torch.arange(x.size(1),device=x.device))
        h=self.encoder(h,src_key_padding_mask=pad)
        return self.head(h.masked_fill(pad[...,None],0).sum(1) / (~pad).sum(1,keepdim=True))
model=ReviewClassifier().to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=.001)
for _ in range(3):
    train_epoch(model,DataLoader(encode(train),batch_size=64,shuffle=True),optimizer)
validation=encode(val)
print(dict(seed=17,device=str(device),accuracy=(predict(model,DataLoader(validation,batch_size=64))==validation.tensors[1]).float().mean().item()))